<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h1 align="center">Perceptron Learning: Training a NAND Gate</h1>

The **perceptron** was introduced in **1958** by *Frank Rosenblatt* and is the simplest neural model that can **learn**.  
Unlike the McCulloch–Pitts neuron, whose weights and threshold are chosen by hand, a perceptron **adjusts its own weights** from labelled examples.

<br>

<div align="center">
    <img src="images/perceptron.png" width="700">
    <br>
    <em><b>A perceptron with two inputs and a bias</b></em>
</div>

<br>

- The neuron computes the net input **net = w₁x₁ + w₂x₂ + b**.
- The net input is compared with a threshold **θ** to produce an output in **{−1, 0, +1}**.
- Whenever the output is wrong, the weights are corrected with the **perceptron learning rule**.
- The **NAND** gate is a *universal gate*: any logic circuit can be built from NAND gates alone, which makes it a good first target for a learning neuron.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Training Data: NAND Truth Table</h2>

The NAND gate outputs **0** only when both inputs are **1**.  
Inputs and targets are written in **bipolar** form (**0 → −1**, **1 → +1**).

- With binary inputs, an input equal to 0 gives **α·t·xᵢ = 0**, so its weight would never change.
- With bipolar inputs, every input takes part in learning.

</div>

In [1]:
# NAND truth table in bipolar encoding: (x1, x2) -> target
NAND_DATASET = [
    ((-1, -1),  1),   # 0 0 -> 1
    ((-1,  1),  1),   # 0 1 -> 1
    (( 1, -1),  1),   # 1 0 -> 1
    (( 1,  1), -1),   # 1 1 -> 0
]

# Show the table in binary and bipolar form
if __name__ == "__main__":
    print("Truth Table:\n")
    print("A  B | NAND || x1  x2 |  t ")
    print("-----------------------------")

    for (x1, x2), t in NAND_DATASET:
        print(f"{(x1+1)//2}  {(x2+1)//2} |  {(t+1)//2}   || {x1:2}  {x2:2} | {t:2}")

Truth Table:

A  B | NAND || x1  x2 |  t 
-----------------------------
0  0 |  1   || -1  -1 |  1
0  1 |  1   || -1   1 |  1
1  0 |  1   ||  1  -1 |  1
1  1 |  0   ||  1   1 | -1


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Activation Function</h2>

The activation function has an **undecided band** of width **2θ** around zero:

- **y = +1** if net > θ  
- **y = 0** if −θ ≤ net ≤ θ  
- **y = −1** if net < −θ

No target is ever 0, so an output inside the band is always counted as an **error**, and it triggers learning.

</div>

In [2]:
# Threshold activation with an undecided band [-theta, theta]
def activation(net, theta):

    if net > theta:
        return 1
    if net < -theta:
        return -1
    return 0

# Test a few net values
if __name__ == "__main__":
    print("net  |  y")
    print("----------")

    for net in [0.5, 0.2, 0.0, -0.2, -0.5]:
        print(f"{net:+.1f} | {activation(net, theta=0.2):+d}")

net  |  y
----------
+0.5 | +1
+0.2 | +0
+0.0 | +0
-0.2 | +0
-0.5 | -1


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Perceptron Learning Rule</h2>

For every training pattern **(x₁, x₂)** with target **t**:

1. Compute **net = w₁x₁ + w₂x₂ + b** and the output **y = activation(net)**.
2. If **y = t**, the weights stay **unchanged**.
3. If **y ≠ t**, update the weights and the bias:  
   - **wᵢ ← wᵢ + α · t · xᵢ**  
   - **b ← b + α · t**

One pass over all patterns is an **epoch**. Training stops when a **full epoch passes without any weight change**.  
Every step is stored in a `history` list so the complete update table can be printed.

</div>

In [3]:
# Perceptron training (pure Python)
def train_perceptron(dataset, theta=0.2, alpha=1.0, b=0.0, w=(0.0, 0.0), max_epochs=100):

    w1, w2 = w
    history = []

    for epoch in range(1, max_epochs + 1):
        changed = False

        for (x1, x2), t in dataset:
            w1_old, w2_old, b_old = w1, w2, b

            net = w1*x1 + w2*x2 + b
            y = activation(net, theta)

            # Update only when the output is wrong
            if y != t:
                dw1, dw2, db = alpha*t*x1, alpha*t*x2, alpha*t
                w1, w2, b = w1 + dw1, w2 + dw2, b + db
                changed = True
            else:
                dw1, dw2, db = 0, 0, 0

            history.append({
                "Epoch": epoch, "x1": x1, "x2": x2, "t": t, "net": net, "y": y, "err": t - y,
                "w1_old": w1_old, "w2_old": w2_old, "b_old": b_old,
                "dw1": dw1, "dw2": dw2, "db": db,
                "w1_new": w1, "w2_new": w2, "b_new": b
            })

        # A full epoch without any change means the training is finished
        if not changed:
            return (w1, w2, b), history, epoch

    raise RuntimeError(f"Did not converge within {max_epochs} epochs "
                       "(the data may not be linearly separable).")


# Prediction and accuracy
def predict(x1, x2, w1, w2, b, theta):
    return activation(w1*x1 + w2*x2 + b, theta)

def accuracy(dataset, w1, w2, b, theta):
    hits = sum(predict(x1, x2, w1, w2, b, theta) == t for (x1, x2), t in dataset)
    return hits / len(dataset)


# Print the training history as a table
COLUMNS = ["Epoch", "x1", "x2", "t", "net", "y", "err",
           "w1_old", "w2_old", "b_old", "dw1", "dw2", "db",
           "w1_new", "w2_new", "b_new"]

def print_training_table(history):
    print(" ".join(f"{c:>6}" for c in COLUMNS))
    print("-" * (7 * len(COLUMNS) - 1))

    for row in history:
        print(" ".join(f"{row[c]:>6g}" for c in COLUMNS))

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Training the NAND Gate</h2>

The experiment uses these parameters:

- Initial weights **w₁ = w₂ = 0**
- Learning rate **α = 1**
- Initial bias **b = 0**
- Threshold **θ = 0.2**

<div align="center">
    <img src="images/learning_curve.png" width="500">
    <br>
    <em><b>Misclassified patterns per epoch</b></em>
</div>

</div>

In [4]:
# Train a perceptron on NAND (theta = 0.2, alpha = 1, b = 0, w = 0)
if __name__ == "__main__":
    (w1, w2, b), history, n_epochs = train_perceptron(NAND_DATASET, theta=0.2, alpha=1, b=0, w=(0, 0))

    print("Weight Update Table:\n")
    print_training_table(history)

    print(f"\nConverged after {n_epochs} epochs")
    print(f"Final weights: w1 = {w1:g}, w2 = {w2:g}, b = {b:g}")

Weight Update Table:

 Epoch     x1     x2      t    net      y    err w1_old w2_old  b_old    dw1    dw2     db w1_new w2_new  b_new
---------------------------------------------------------------------------------------------------------------
     1     -1     -1      1      0      0      1      0      0      0     -1     -1      1     -1     -1      1
     1     -1      1      1      1      1      0     -1     -1      1      0      0      0     -1     -1      1
     1      1     -1      1      1      1      0     -1     -1      1      0      0      0     -1     -1      1
     1      1      1     -1     -1     -1      0     -1     -1      1      0      0      0     -1     -1      1
     2     -1     -1      1      3      1      0     -1     -1      1      0      0      0     -1     -1      1
     2     -1      1      1      1      1      0     -1     -1      1      0      0      0     -1     -1      1
     2      1     -1      1      1      1      0     -1     -1      1      0      

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Reading the Table (First Step by Hand)</h2>

**Pattern 1:** (x₁, x₂) = (−1, −1), t = +1, with w₁ = w₂ = b = 0

- net = 0·(−1) + 0·(−1) + 0 = **0**
- Since −0.2 ≤ 0 ≤ 0.2, the output is **y = 0 ≠ t**, so this is an **error**.
- Δw₁ = α·t·x₁ = **−1**, Δw₂ = **−1**, Δb = α·t = **+1**
- New values: **w₁ = −1, w₂ = −1, b = 1**

The other three patterns are classified correctly with these weights, so nothing changes.  
Epoch 2 only **verifies** this: it finishes without an error, so training stops.

</div>

In [5]:
# Verify the trained neuron on the full truth table
if __name__ == "__main__":
    print("Truth Table Check:\n")
    print("x1  x2 |  net |  t |  y | OK")
    print("-----------------------------")

    for (x1, x2), t in NAND_DATASET:
        net = w1*x1 + w2*x2 + b
        y = predict(x1, x2, w1, w2, b, theta=0.2)
        print(f"{x1:2}  {x2:2} | {net:4g} | {t:2} | {y:2} | {'yes' if y == t else 'NO'}")

    print(f"\nAccuracy = {accuracy(NAND_DATASET, w1, w2, b, theta=0.2):.0%}")

Truth Table Check:

x1  x2 |  net |  t |  y | OK
-----------------------------
-1  -1 |    3 |  1 |  1 | yes
-1   1 |    1 |  1 |  1 | yes
 1  -1 |    1 |  1 |  1 | yes
 1   1 |   -1 | -1 | -1 | yes

Accuracy = 100%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Decision Boundary</h2>

The trained neuron separates the inputs with the line **net = 0**, that is **−x₁ − x₂ + 1 = 0**.

- The solid line is **net = 0**.
- The dashed lines are **net = ±θ**, and the blue band between them is the **undecided zone**.
- The three patterns with **t = +1** lie on one side and the single pattern with **t = −1** lies on the other side.

<div align="center">
    <img src="images/decision_boundary.png" width="600">
</div>

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Is the Solution Always the Same?</h2>

**No.** NAND is **linearly separable**, so the **Perceptron Convergence Theorem** guarantees that training stops.  
But *which* weights it finds depends on:

- the initial **bias** and the threshold **θ**
- the **learning rate α**
- the initial **weights**
- the **order** of the training patterns

The next cells check this experimentally.

</div>

In [6]:
# Change theta, alpha and the initial bias
if __name__ == "__main__":
    configs = [(0.2, 1, 0), (0, 1, 0), (0.5, 1, 0), (1, 1, 1),
               (0.2, 1, -1), (0.2, 0.5, 0), (0.2, 2, 0), (1.5, 1, 0)]

    print("theta  alpha  b0 | epochs |  w1    w2     b  | accuracy")
    print("-------------------------------------------------------")

    for theta, alpha, b0 in configs:
        (w1, w2, b), _, n_epochs = train_perceptron(NAND_DATASET, theta=theta, alpha=alpha, b=b0)
        acc = accuracy(NAND_DATASET, w1, w2, b, theta)
        print(f"{theta:5g} {alpha:6g} {b0:3g} | {n_epochs:6} | {w1:5g} {w2:5g} {b:5g} | {acc:7.0%}")

theta  alpha  b0 | epochs |  w1    w2     b  | accuracy
-------------------------------------------------------
  0.2      1   0 |      2 |    -1    -1     1 |    100%
    0      1   0 |      2 |    -1    -1     1 |    100%
  0.5      1   0 |      2 |    -1    -1     1 |    100%
    1      1   1 |      3 |    -3    -3     2 |    100%
  0.2      1  -1 |      2 |    -2    -2     1 |    100%
  0.2    0.5   0 |      2 |  -0.5  -0.5   0.5 |    100%
  0.2      2   0 |      2 |    -2    -2     2 |    100%
  1.5      1   0 |      2 |    -2    -2     2 |    100%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

Different settings give different weights, and **every one of them classifies NAND correctly**:

<div align="center">
    <img src="images/boundaries.png" width="1000">
</div>

</div>

In [7]:
# Random initial weights, random bias and random pattern order
import random

if __name__ == "__main__":
    random.seed(42)
    solutions = set()

    for _ in range(200):
        w0 = (random.randint(-2, 2), random.randint(-2, 2))
        b0 = random.randint(-2, 2)
        data = random.sample(NAND_DATASET, k=len(NAND_DATASET))   # shuffled copy

        (w1, w2, b), _, _ = train_perceptron(data, theta=0.2, alpha=1, b=b0, w=w0)
        assert accuracy(NAND_DATASET, w1, w2, b, theta=0.2) == 1.0
        solutions.add((w1, w2, b))

    print(f"{len(solutions)} different solutions were found in 200 runs (all with 100% accuracy).\n")
    print("Some of them:")
    for w1, w2, b in sorted(solutions)[:8]:
        print(f"  w1 = {w1:g}, w2 = {w2:g}, b = {b:g}")

5 different solutions were found in 200 runs (all with 100% accuracy).

Some of them:
  w1 = -2, w2 = -2, b = 1
  w1 = -2, w2 = -2, b = 2
  w1 = -2, w2 = -1, b = 2
  w1 = -1, w2 = -2, b = 2
  w1 = -1, w2 = -1, b = 1


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">The XOR Limitation</h2>

A single perceptron can only learn **linearly separable** functions.  
**XOR** is not one of them: no straight line can separate **{(−1,−1), (1,1)}** from **{(−1,1), (1,−1)}**.

<div align="center">
    <img src="images/xor.png" width="500">
</div>

The learning rule never reaches an error-free epoch, so training does not converge:

</div>

In [8]:
# XOR cannot be learned by a single perceptron
XOR_DATASET = [
    ((-1, -1), -1),
    ((-1,  1),  1),
    (( 1, -1),  1),
    (( 1,  1), -1),
]

if __name__ == "__main__":
    try:
        train_perceptron(XOR_DATASET, theta=0.2, max_epochs=50)
    except RuntimeError as e:
        print("Training failed:", e)

Training failed: Did not converge within 50 epochs (the data may not be linearly separable).


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

XOR needs a **hidden layer**: two hidden neurons compute **x₁ AND NOT x₂** and **x₂ AND NOT x₁**, and an output neuron combines them with **OR**.  
This is exactly the multi-layer idea that leads to modern deep learning.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>Conclusion</h2>
In this notebook, we trained a single **perceptron** to learn the **NAND** gate using only pure Python.

Through our experiments, we demonstrated:
- **Learning from Errors:** With the perceptron rule, weights change only when the output is wrong. With θ = 0.2, α = 1 and zero initial values, the neuron converges after **2 epochs** to **w₁ = w₂ = −1, b = 1**.
- **Non-unique Solutions:** The final weights depend on θ, α, the initial values and the pattern order, but every solution separates NAND correctly.
- **Historical Limitations:** A single perceptron cannot learn **XOR**, because it is not linearly separable. This limitation motivated **Multi-layer Perceptrons** and, later, **Deep Learning**.

Understanding this one neuron makes it much easier to follow how larger networks learn.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>👩‍💻 About the Author</h2>

<b>Sarah M Afshar</b>  
<i>Python Developer | Generative AI Specialist</i>

I am a passionate developer focused on the intersection of <b>Large Language Models (LLMs)</b>, neural network architectures, and the evolution of intelligent systems.  
This project is part of my journey in exploring the mathematical foundations of AI.

🔗 GitHub: https://github.com/afshars  

🔗 LinkedIn: https://www.linkedin.com/in/sara-m-afshar/

</div>